## M5 Dataset Information
`datasetsforecast` returns three related tables:

- `Y_df`: the target sales history
- `X_df`: time-varying features such as sell price, events, and SNAP indicators
- `S_df`: static item/store/category metadata

We will inspect them separately first, then join them into one analytical table called `m5`.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from datasetsforecast.m5 import M5

In [2]:
Y_df, X_df, S_df = M5.load(directory="data", cache=True)

In [3]:
print("Y_df:", Y_df.shape)
print("X_df:", X_df.shape)
print("S_df:", S_df.shape)

Y_df: (47649940, 3)
X_df: (47649940, 10)
S_df: (30490, 6)


In [4]:
Y_df.head()


,unique_id,ds,y
0,FOODS_1_001_CA_1,2011-01-29,3.0
1,FOODS_1_001_CA_1,2011-01-30,0.0
2,FOODS_1_001_CA_1,2011-01-31,0.0
3,FOODS_1_001_CA_1,2011-02-01,1.0
4,FOODS_1_001_CA_1,2011-02-02,4.0


In [5]:
X_df.head()

,unique_id,ds,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_001_CA_1,2011-01-29,nan,nan,nan,nan,0,0,0,2.0
1,FOODS_1_001_CA_1,2011-01-30,nan,nan,nan,nan,0,0,0,2.0
2,FOODS_1_001_CA_1,2011-01-31,nan,nan,nan,nan,0,0,0,2.0
3,FOODS_1_001_CA_1,2011-02-01,nan,nan,nan,nan,1,1,0,2.0
4,FOODS_1_001_CA_1,2011-02-02,nan,nan,nan,nan,1,0,1,2.0


In [6]:
S_df.head()


,unique_id,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_1_001_CA_1,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1969,FOODS_1_001_CA_2,FOODS_1_001,FOODS_1,FOODS,CA_2,CA
3938,FOODS_1_001_CA_3,FOODS_1_001,FOODS_1,FOODS,CA_3,CA
5907,FOODS_1_001_CA_4,FOODS_1_001,FOODS_1,FOODS,CA_4,CA
7875,FOODS_1_001_TX_1,FOODS_1_001,FOODS_1,FOODS,TX_1,TX


In [7]:
print("Date range:", Y_df["ds"].min(), "to", Y_df["ds"].max())
print("Number of unique series:", Y_df["unique_id"].nunique())
print("Rows:", len(Y_df))
print("Duplicate unique_id-date rows:", Y_df.duplicated(subset=["unique_id", "ds"]).sum())
print("Negative sales rows:", (Y_df["y"] < 0).sum())
print("Missing target values:", Y_df["y"].isna().sum())

Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Number of unique series: 30490
Rows: 47649940
Duplicate unique_id-date rows: 0
Negative sales rows: 0
Missing target values: 0


In [8]:
m5 = (
    Y_df
    .merge(X_df, on=["unique_id", "ds"], how="left")
    .merge(S_df, on="unique_id", how="left")
)

print("m5 shape:", m5.shape)
m5.head()

m5 shape: (47649940, 16)


,unique_id,ds,y,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,item_id,dept_id,cat_id,store_id,state_id
0,FOODS_1_001_CA_1,2011-01-29,3.0,nan,nan,nan,nan,0,0,0,2.0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
1,FOODS_1_001_CA_1,2011-01-30,0.0,nan,nan,nan,nan,0,0,0,2.0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
2,FOODS_1_001_CA_1,2011-01-31,0.0,nan,nan,nan,nan,0,0,0,2.0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
3,FOODS_1_001_CA_1,2011-02-01,1.0,nan,nan,nan,nan,1,1,0,2.0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA
4,FOODS_1_001_CA_1,2011-02-02,4.0,nan,nan,nan,nan,1,0,1,2.0,FOODS_1_001,FOODS_1,FOODS,CA_1,CA


In [9]:
print("Date range:", Y_df["ds"].min(), "to", Y_df["ds"].max())
print("Date range:", m5["ds"].min(), "to", m5["ds"].max())
print("Rows before merge:", len(Y_df))
print("Rows after merge:", len(m5))
print("Duplicate series-date rows:", m5.duplicated(["unique_id", "ds"]).sum())
print("Missing categories:", m5["cat_id"].isna().sum())
print("Missing departments:", m5["dept_id"].isna().sum())

Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Rows before merge: 47649940
Rows after merge: 47649940
Duplicate series-date rows: 0
Missing categories: 0
Missing departments: 0


In [10]:
sales_key = ["unique_id", "ds"]

print("Expected rows:", len(Y_df))
print("Actual rows:", len(m5))
print("Row difference:", len(m5) - len(Y_df))
print("Duplicate sales keys:", m5.duplicated(sales_key).sum())

expected_columns = (
    set(Y_df.columns)
    | (set(S_df.columns) - {"unique_id"})
    | (set(X_df.columns) - {"ds"})
)

print("Missing expected columns:", expected_columns - set(m5.columns))
print("Unexpected columns:", set(m5.columns) - expected_columns)

Expected rows: 47649940
Actual rows: 47649940
Row difference: 0
Duplicate sales keys: 0
Missing expected columns: set()
Unexpected columns: set()


In [11]:
assert len(m5) == len(Y_df)
assert m5.duplicated(sales_key).sum() == 0
assert expected_columns == set(m5.columns)

In [12]:
print("Total missing values:", m5.isna().sum().sum())

Total missing values: 0


In [16]:
m5.to_parquet("data/processed/category_daily_raw.parquet", index=False)

## Data Preparation Conclusion

The M5 target, calendar, price, and item metadata tables were joined into one item-store-day analytical dataset. The join checks passed with no duplicate sales keys, no missing expected columns, and no missing values. The saved dataset is used in the next notebook to create category-level daily demand views.